# **DeBERTa**

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1" # "0" o "1"

In [2]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [3]:
from utils import *

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)

/home/n.emmolo/miniconda3/envs/env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-11-08 15:44:26.801921: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-08 15:44:26.863938: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-08 15:44:28.157898: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may 

In [4]:
# Ignore warnings for cleaner output
import warnings
warnings.filterwarnings("ignore")

In [5]:
# --------------------
# Build model function
# --------------------

def build_model(learning_rate=2e-5, weight_decay=0.1):
    """
    Builds a DeBERTa model (microsoft/deberta-base) for sequence classification.

    Args:
        learning_rate (float): Learning rate for the optimizer.
        weight_decay (float): Weight decay for AdamW optimizer.

    Returns:
        model (AutoModelForSequenceClassification): HuggingFace DeBERTa model.
        tokenizer (AutoTokenizer): Tokenizer associated with the model.
        train_args (TrainingArguments): Default training configuration.
    """

    model = AutoModelForSequenceClassification.from_pretrained("microsoft/deberta-base", num_labels=2)
    tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-base")

    train_args = TrainingArguments(
        output_dir="./deberta_finetune_output",
        eval_strategy="epoch",
        save_strategy="no",
        learning_rate=learning_rate,
        weight_decay=weight_decay,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=3,
        logging_dir="./logs",
        load_best_model_at_end=False,
        logging_steps=50,
        seed=42,
    )

    return model, tokenizer, train_args

In [6]:
# ----------------------
# Tokenization functions
# ----------------------

def tokenize_function(examples, tokenizer, max_len=128):
    """
    Tokenizes the input examples using the provided tokenizer.

    Args:
        examples (dict): A dictionary containing the text data to be tokenized.
        tokenizer (AutoTokenizer): The tokenizer to use for tokenization.
        max_len (int): Maximum length for padding/truncation.

    Returns:
        dict: Tokenized inputs with padding and truncation applied.
    """

    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=max_len,
    )


def tokenize_datasets(datasets, tokenizer):
    """
    Tokenizes multiple datasets using the provided tokenizer.

    Args:
        datasets (dict): A dictionary where keys are dataset names and values are dictionaries with 'train', 'val', and 'test' splits.
        tokenizer (AutoTokenizer): The tokenizer to use for tokenization.

    Returns:
        dict: A dictionary with the same keys as input datasets, but with tokenized datasets.
    """

    tokenized_datasets = {}
    
    for name, data in datasets.items():
        print(f"\n=== Tokenizing dataset: {name} ===")

        train_dataset = Dataset.from_dict({"text": data["train"][0], "label": data["train"][1].astype(int)})
        val_dataset = Dataset.from_dict({"text": data["val"][0], "label": data["val"][1].astype(int)})
        test_dataset = Dataset.from_dict({"text": data["test"][0], "label": data["test"][1].astype(int)})

        train_tokenized = train_dataset.map(lambda x: tokenize_function(x, tokenizer), batched=True)
        val_tokenized = val_dataset.map(lambda x: tokenize_function(x, tokenizer), batched=True)
        test_tokenized = test_dataset.map(lambda x: tokenize_function(x, tokenizer), batched=True)

        tokenized_datasets[name] = {
            "train": (train_tokenized, np.array(train_dataset["label"])),
            "val": (val_tokenized, np.array(val_dataset["label"])),
            "test": (test_tokenized, np.array(test_dataset["label"]))
        }

    return tokenized_datasets

## VERSION 1: Dataset (Simple)

In [7]:
dataset_df = data_loading() # load all datasets

for name, df in dataset_df.items():
    print(f"Name: {name}, Number of samples: {len(df)}")

# dataset_df = dict(list(dataset_df.items())[:5])

print("\nSplitting datasets into train/val/test...")
datasets = {name: split_dataset(df) for name, df in dataset_df.items()} # split all datasets in train/val/test

model, tokenizer, train_args = build_model()

print("\nComputing tokenized datasets...")
datasets =  tokenize_datasets(datasets, tokenizer) # tokenize all datasets

Name: Celebrity, Number of samples: 500
Name: CIDII, Number of samples: 722
Name: FaKES, Number of samples: 842
Name: FakeVsSatire, Number of samples: 486
Name: Horne, Number of samples: 326
Name: Infodemic, Number of samples: 10559
Name: ISOT, Number of samples: 44271
Name: Kaggle_clement, Number of samples: 39105
Name: Kaggle_meg, Number of samples: 12845
Name: LIAR_PLUS, Number of samples: 12784
Name: Politifact, Number of samples: 504
Name: Unipi_NDF, Number of samples: 554

Splitting datasets into train/val/test...


Some weights of DebertaForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Computing tokenized datasets...

=== Tokenizing dataset: Celebrity ===


Map: 100%|███████████████████████████████████████████████████████████████| 100/100 [00:00<00:00, 1893.22 examples/s]



=== Tokenizing dataset: CIDII ===


...

Map: 100%|████████████████████████████████████████████████████████████| 2557/2557 [00:00<00:00, 11629.67 examples/s]



=== Tokenizing dataset: Politifact ===


Map: 100%|████████████████████████████████████████████████████████████████| 101/101 [00:00<00:00, 785.23 examples/s]



=== Tokenizing dataset: Unipi_NDF ===


Map: 100%|███████████████████████████████████████████████████████████████| 111/111 [00:00<00:00, 3758.22 examples/s]


In [8]:
# --------------------------------
# Fine-tuning on multiple datasets
# --------------------------------

results = {}

for i, (name, data) in enumerate(datasets.items()):
    print(f"\n=== Phase {i+1}: Fine-tuning on {name} ===")

    X_train, y_train = data["train"]
    X_val, y_val = data["val"]
    X_test, y_test = data["test"]

    # define trainer
    trainer = Trainer(
        model=model,
        args=train_args,
        train_dataset=X_train,
        eval_dataset=X_val,
    )

    # fine-tune on train + val
    trainer.train()

    # evaluate on current dataset
    y_pred = trainer.predict(X_test)
    y_pred = np.argmax(y_pred.predictions, axis=1)
    print(f"\nClassification Report after {name}:")
    print(classification_report(y_test, y_pred))
    print(f"Confusion Matrix after {name}:")
    print(confusion_matrix(y_test, y_pred))
    print(f"Weighted F1-score after {name}: {f1_score(y_test, y_pred, average='weighted'):.4f}")

    # evaluate on all datasets
    print("\n--- Evaluation on all datasets ---")
    results[name] = {}
    for test_name, test_data in datasets.items():
        X_te, y_te = test_data["test"]
        preds = trainer.predict(X_te)
        preds = np.argmax(preds.predictions, axis=1)
        f1 = f1_score(y_te, preds, average="weighted")
        results[name][test_name] = f1
        print(f"Evaluation on {test_name}: Weighted F1 = {f1:.4f}")



=== Phase 1: Fine-tuning on Celebrity ===


Epoch,Training Loss,Validation Loss
1,No log,0.653105
2,0.643900,0.849022
3,0.506800,0.750181



Classification Report after Celebrity:
              precision    recall  f1-score   support

           0       0.74      0.86      0.80        50
           1       0.83      0.70      0.76        50

    accuracy                           0.78       100
   macro avg       0.79      0.78      0.78       100
weighted avg       0.79      0.78      0.78       100

Confusion Matrix after Celebrity:
[[43  7]
 [15 35]]
Weighted F1-score after Celebrity: 0.7786

--- Evaluation on all datasets ---


Evaluation on Celebrity: Weighted F1 = 0.7786


Evaluation on CIDII: Weighted F1 = 0.8288


...

Evaluation on Kaggle_clement: Weighted F1 = 0.9088


Evaluation on Kaggle_meg: Weighted F1 = 0.5889


Evaluation on LIAR_PLUS: Weighted F1 = 0.5546


Evaluation on Politifact: Weighted F1 = 0.8743


Evaluation on Unipi_NDF: Weighted F1 = 0.8612


In [9]:
# ---------------
# Results summary
# ---------------

print("\n=== Results Summary ===")
for name, res in results.items():
    print(f"\nResults after training on {name}:")
    for test_name, f1 in res.items():
        print(f"  Test on {test_name}: Weighted F1 = {f1:.4f}")


=== Results Summary ===

Results after training on Celebrity:
  Test on Celebrity: Weighted F1 = 0.7786
  Test on CIDII: Weighted F1 = 0.8288
  Test on FaKES: Weighted F1 = 0.4325
  Test on FakeVsSatire: Weighted F1 = 0.6363
  Test on Horne: Weighted F1 = 0.6390
  Test on Infodemic: Weighted F1 = 0.3982
  Test on ISOT: Weighted F1 = 0.7367
  Test on Kaggle_clement: Weighted F1 = 0.9157
  Test on Kaggle_meg: Weighted F1 = 0.3936
  Test on LIAR_PLUS: Weighted F1 = 0.4752
  Test on Politifact: Weighted F1 = 0.6983
  Test on Unipi_NDF: Weighted F1 = 0.6190

Results after training on CIDII:
  Test on Celebrity: Weighted F1 = 0.3552
  Test on CIDII: Weighted F1 = 0.9653
  Test on FaKES: Weighted F1 = 0.3388
  Test on FakeVsSatire: Weighted F1 = 0.4278
  Test on Horne: Weighted F1 = 0.5500
  Test on Infodemic: Weighted F1 = 0.3072
  Test on ISOT: Weighted F1 = 0.9232
  Test on Kaggle_clement: Weighted F1 = 0.5403
  Test on Kaggle_meg: Weighted F1 = 0.0102
  Test on LIAR_PLUS: Weighted F1 = 0

## VERSION 2: Dataset by Topic

In [7]:
dataset_df = data_by_topic()

# split "politics" in "politics1" and "politics2"
if "politics" in dataset_df:
    dataset_df["politics1"] = dataset_df["politics"].sample(frac=0.5, random_state=42)
    dataset_df["politics2"] = dataset_df["politics"].drop(dataset_df["politics1"].index)
    # drop original "politics" dataset
    del dataset_df["politics"]
    # put "politics1" and "politics2" at the beginning of the dict
    dataset_df = {"politics1": dataset_df["politics1"], "politics2": dataset_df["politics2"], **dataset_df}

for topic, df in dataset_df.items():
    print(f"Topic: {topic}, Number of samples: {len(df)}")

# dataset_df = dict(list(dataset_df.items())[3:]) # remove first 3 datasets

print("\nSplitting datasets into train/val/test...")
datasets = {topic: split_dataset(df) for topic, df in dataset_df.items()} # split all datasets in train/val/test

model, tokenizer, train_args = build_model()

print("\nComputing tokenized datasets...")
datasets =  tokenize_datasets(datasets, tokenizer) # tokenize all datasets

Topic: politics1, Number of samples: 48738
Topic: politics2, Number of samples: 48738
Topic: general, Number of samples: 12845
Topic: covid, Number of samples: 10559
Topic: syria, Number of samples: 842
Topic: islam, Number of samples: 722
Topic: notredame, Number of samples: 554
Topic: gossip, Number of samples: 500

Splitting datasets into train/val/test...


Some weights of DebertaForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Computing tokenized datasets...

=== Tokenizing dataset: politics1 ===


Map: 100%|█████████████████████████████████████████████████████████████| 9748/9748 [00:03<00:00, 2843.11 examples/s]



=== Tokenizing dataset: politics2 ===


...

Map: 100%|███████████████████████████████████████████████████████████████| 145/145 [00:00<00:00, 4496.83 examples/s]



=== Tokenizing dataset: notredame ===


Map: 100%|███████████████████████████████████████████████████████████████| 111/111 [00:00<00:00, 4855.43 examples/s]



=== Tokenizing dataset: gossip ===


Map: 100%|███████████████████████████████████████████████████████████████| 100/100 [00:00<00:00, 2238.11 examples/s]


In [8]:
# -------------------------------
# Fine-tuning on Dataset by Topic
# -------------------------------

results = {}

# sequential training
for i, (topic, data) in enumerate(datasets.items()):
    print(f"\n=== Phase {i+1}: Training/Fine-tuning on topic: {topic} ===")

    X_train, y_train = data["train"]
    X_val, y_val = data["val"]
    X_test, y_test = data["test"]

    # define trainer
    trainer = Trainer(
        model=model,
        args=train_args,
        train_dataset=X_train,
        eval_dataset=X_val,
    )

    # fine-tune on train + val
    trainer.train()

    # evaluate on current dataset
    y_pred = trainer.predict(X_test)
    y_pred = np.argmax(y_pred.predictions, axis=1)
    print(f"\nClassification Report after topic {topic}:")
    print(classification_report(y_test, y_pred))
    print(f"Confusion Matrix after topic {topic}:")
    print(confusion_matrix(y_test, y_pred))
    print(f"Weighted F1-score after topic {topic}:", f1_score(y_test, y_pred, average='weighted'))

    # evaluate on all datasets
    print("\n--- Evaluation on all datasets ---")
    results[topic] = {}
    for test_topic, test_data in datasets.items(): # for each topic
        X_te, y_te = test_data["test"]
        preds = trainer.predict(X_te)
        preds = np.argmax(preds.predictions, axis=1)
        f1 = f1_score(y_te, preds, average="weighted")
        results[topic][test_topic] = f1
        print(f"Evaluation on topic {test_topic}: Weighted F1 = {f1:.4f}")


=== Phase 1: Training/Fine-tuning on topic: politics1 ===


Epoch,Training Loss,Validation Loss
1,0.131900,0.137567
2,0.090600,0.127495
3,0.096300,0.181145



Classification Report after topic politics1:
              precision    recall  f1-score   support

           0       0.93      0.97      0.95      5031
           1       0.96      0.93      0.94      4717

    accuracy                           0.95      9748
   macro avg       0.95      0.95      0.95      9748
weighted avg       0.95      0.95      0.95      9748

Confusion Matrix after topic politics1:
[[4863  168]
 [ 352 4365]]
Weighted F1-score after topic politics1: 0.9466041492905594

--- Evaluation on all datasets ---


Evaluation on topic politics1: Weighted F1 = 0.9466


Evaluation on topic politics2: Weighted F1 = 0.9470


...

Evaluation on topic covid: Weighted F1 = 0.9169


Evaluation on topic syria: Weighted F1 = 0.5014


Evaluation on topic islam: Weighted F1 = 0.7847


Evaluation on topic notredame: Weighted F1 = 0.8699


Evaluation on topic gossip: Weighted F1 = 0.7993


In [9]:
# ---------------
# Results summary
# ---------------

print("\n=== Results Summary ===")
for topic, res in results.items():
    print(f"\nResults after training on topic {topic}:")
    for test_topic, f1 in res.items():
        print(f"  Test on topic {test_topic}: Weighted F1 = {f1:.4f}")


=== Results Summary ===

Results after training on topic politics1:
  Test on topic politics1: Weighted F1 = 0.9466
  Test on topic politics2: Weighted F1 = 0.9470
  Test on topic general: Weighted F1 = 0.3416
  Test on topic covid: Weighted F1 = 0.3986
  Test on topic syria: Weighted F1 = 0.5307
  Test on topic islam: Weighted F1 = 0.3568
  Test on topic notredame: Weighted F1 = 0.2571
  Test on topic gossip: Weighted F1 = 0.5746

Results after training on topic politics2:
  Test on topic politics1: Weighted F1 = 0.9473
  Test on topic politics2: Weighted F1 = 0.9493
  Test on topic general: Weighted F1 = 0.3831
  Test on topic covid: Weighted F1 = 0.4735
  Test on topic syria: Weighted F1 = 0.5180
  Test on topic islam: Weighted F1 = 0.4118
  Test on topic notredame: Weighted F1 = 0.3756
  Test on topic gossip: Weighted F1 = 0.4615

Results after training on topic general:
  Test on topic politics1: Weighted F1 = 0.3514
  Test on topic politics2: Weighted F1 = 0.3552
  Test on topic

## VERSION 3: Dataset by Date

In [7]:
dataset_df = data_by_date()

for date, df in dataset_df.items():
    print(f"Date: {date}, Number of samples: {len(df)}")

# dataset_df = dict(list(dataset_df.items())[:3]) # remove last 3 datasets

print("\nSplitting datasets into train/val/test...")
datasets = {date: split_dataset(df) for date, df in dataset_df.items()} # split all datasets in train/val/test

model, tokenizer, train_args = build_model()

print("\nComputing tokenized datasets...")
datasets =  tokenize_datasets(datasets, tokenizer) # tokenize all datasets

Date: 2011-2013, Number of samples: 55
Date: 2014, Number of samples: 114
Date: 2015, Number of samples: 84
Date: 2016, Number of samples: 63018
Date: 2017, Number of samples: 16657
Date: 2019, Number of samples: 554
Date: 2020, Number of samples: 10559

Splitting datasets into train/val/test...


Some weights of DebertaForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Computing tokenized datasets...

=== Tokenizing dataset: 2011-2013 ===


Map: 100%|██████████████████████████████████████████████████████████████████| 11/11 [00:00<00:00, 480.23 examples/s]


=== Tokenizing dataset: 2014 ===


...

Map: 100%|█████████████████████████████████████████████████████████████| 3332/3332 [00:01<00:00, 3092.70 examples/s]



=== Tokenizing dataset: 2019 ===


Map: 100%|███████████████████████████████████████████████████████████████| 111/111 [00:00<00:00, 3167.82 examples/s]



=== Tokenizing dataset: 2020 ===


Map: 100%|█████████████████████████████████████████████████████████████| 2112/2112 [00:00<00:00, 9961.89 examples/s]


In [8]:
# ------------------------------
# Fine-tuning on Dataset by Date
# ------------------------------

results = {}

# sequential training
for i, (date, data) in enumerate(datasets.items()):
    print(f"\n=== Phase {i+1}: Training/Fine-tuning on date: {date} ===")
    
    X_train, y_train = data["train"]
    X_val, y_val = data["val"]
    X_test, y_test = data["test"]

    # define trainer
    trainer = Trainer(
        model=model,
        args=train_args,
        train_dataset=X_train,
        eval_dataset=X_val,
    )
    
    # fine-tune on train + val
    trainer.train()

    # evaluate on current dataset
    y_pred = trainer.predict(X_test)
    y_pred = np.argmax(y_pred.predictions, axis=1)
    print(f"\nClassification Report after date {date}:")
    print(classification_report(y_test, y_pred))
    print(f"Confusion Matrix after date {date}:")
    print(confusion_matrix(y_test, y_pred))
    print(f"Weighted F1-score after date {date}:", f1_score(y_test, y_pred, average='weighted'))

    # evaluate on all datasets
    print("\n--- Evaluation on all datasets ---")
    results[date] = {}
    for test_date, test_data in datasets.items(): # for each date
        X_te, y_te = test_data["test"]
        preds = trainer.predict(X_te)
        preds = np.argmax(preds.predictions, axis=1)
        f1 = f1_score(y_te, preds, average="weighted")
        results[date][test_date] = f1
        print(f"Evaluation on date {test_date}: Weighted F1 = {f1:.4f}")


=== Phase 1: Training/Fine-tuning on date: 2011-2013 ===


Epoch,Training Loss,Validation Loss
1,No log,0.690401
2,No log,0.690204
3,No log,0.690555



Classification Report after date 2011-2013:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         5
           1       0.55      1.00      0.71         6

    accuracy                           0.55        11
   macro avg       0.27      0.50      0.35        11
weighted avg       0.30      0.55      0.39        11

Confusion Matrix after date 2011-2013:
[[0 5]
 [0 6]]
Weighted F1-score after date 2011-2013: 0.3850267379679144

--- Evaluation on all datasets ---


Evaluation on date 2011-2013: Weighted F1 = 0.3850


Evaluation on date 2014: Weighted F1 = 0.3095


...

Evaluation on date 2015: Weighted F1 = 0.3665


Evaluation on date 2016: Weighted F1 = 0.7929


Evaluation on date 2017: Weighted F1 = 0.9960


Evaluation on date 2019: Weighted F1 = 0.6062


Evaluation on date 2020: Weighted F1 = 0.9787


In [9]:
# ---------------
# Results summary
# ---------------

print("\n=== Results Summary ===")
for date, res in results.items():
    print(f"\nResults after training on date {date}:")
    for test_date, f1 in res.items():
        print(f"  Test on date {test_date}: Weighted F1 = {f1:.4f}")


=== Results Summary ===

Results after training on date 2011-2013:
  Test on date 2011-2013: Weighted F1 = 0.3850
  Test on date 2014: Weighted F1 = 0.3095
  Test on date 2015: Weighted F1 = 0.3665
  Test on date 2016: Weighted F1 = 0.2058
  Test on date 2017: Weighted F1 = 0.0000
  Test on date 2019: Weighted F1 = 0.2163
  Test on date 2020: Weighted F1 = 0.3074

Results after training on date 2014:
  Test on date 2011-2013: Weighted F1 = 0.3850
  Test on date 2014: Weighted F1 = 0.3095
  Test on date 2015: Weighted F1 = 0.3665
  Test on date 2016: Weighted F1 = 0.2058
  Test on date 2017: Weighted F1 = 0.0000
  Test on date 2019: Weighted F1 = 0.2163
  Test on date 2020: Weighted F1 = 0.3074

Results after training on date 2015:
  Test on date 2011-2013: Weighted F1 = 0.3850
  Test on date 2014: Weighted F1 = 0.3095
  Test on date 2015: Weighted F1 = 0.3665
  Test on date 2016: Weighted F1 = 0.2058
  Test on date 2017: Weighted F1 = 0.0000
  Test on date 2019: Weighted F1 = 0.2163
 